# 03 — Indexing & Slicing
**Goal:** Select any subset of an array — single values, rows, columns, arbitrary masks.

```
arr[i]          → element at position i
arr[i, j]       → element at row i, column j  (2D)
arr[start:stop] → slice from start to stop-1
arr[mask]       → boolean selection
```

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd

## 1. 1D indexing

In [ ]:
sessions = np.array([3200, 2800, 3500, 2600, 4100])

print(sessions[0])    # first element
print(sessions[-1])   # last element
print(sessions[1:4])  # slice [2800, 3500, 2600]
print(sessions[:3])   # first 3
print(sessions[::2])  # every other element

## 2. 2D indexing — rows and columns

In [ ]:
# Funnel: rows = metric, cols = channel (organic,paid,email,social,direct)
funnel = np.array([
    [3200, 2800, 3500, 2600, 4100],   # row 0: sessions
    [2400, 1960, 2800, 1820, 3280],   # row 1: starts
    [  96,   59,  158,   47,  156],   # row 2: activations
])

print('Element [0, 2]:  ', funnel[0, 2])   # sessions of email channel
print('Row 0 (sessions):', funnel[0])       # full row
print('Col 0 (organic): ', funnel[:, 0])    # full column
print()
# Submatrix: rows 0-1, cols 1-3
print(funnel[0:2, 1:4])

In [ ]:
# Last row and last column
print('Last row:   ', funnel[-1])       # activations
print('Last col:   ', funnel[:, -1])    # direct channel

## 3. Boolean indexing — filter by condition

In [ ]:
sessions = np.array([3200, 2800, 3500, 2600, 4100])
channels = np.array(['organic', 'paid', 'email', 'social', 'direct'])

# Which channels had more than 3000 sessions?
mask = sessions > 3000
print('mask:     ', mask)              # [True, False, True, False, True]
print('channels: ', channels[mask])    # organic, email, direct
print('sessions: ', sessions[mask])    # 3200, 3500, 4100

In [ ]:
# Compound conditions — use & | ~  (not 'and', 'or', 'not')
cvr = np.array([3.0, 2.1, 4.5, 1.8, 3.8])

# High sessions AND decent CVR
good = (sessions > 3000) & (cvr > 3.0)
print('High-perf channels:', channels[good])

# Below 3000 sessions OR CVR below 2
concern = (sessions < 3000) | (cvr < 2.0)
print('Concern channels:  ', channels[concern])

In [ ]:
# np.where — conditional assignment (vectorized if-else)
# np.where(condition, value_if_true, value_if_false)
label = np.where(sessions > 3000, 'high', 'low')
print(label)  # ['high', 'low', 'high', 'low', 'high']

## 4. Fancy indexing — select by index list

In [ ]:
sessions = np.array([3200, 2800, 3500, 2600, 4100])

# Pick specific positions
idx = [0, 2, 4]          # organic, email, direct
print(sessions[idx])     # [3200, 3500, 4100]

# Order matters — and you can repeat indices
print(sessions[[4, 0, 4]])  # [4100, 3200, 4100]

In [ ]:
# 2D fancy indexing
funnel = np.array([
    [3200, 2800, 3500, 2600, 4100],
    [2400, 1960, 2800, 1820, 3280],
    [  96,   59,  158,   47,  156],
])

# Select rows 0 and 2 (sessions and activations only)
print(funnel[[0, 2]])

## 5. Views vs copies — a critical gotcha

In [ ]:
arr = np.array([10, 20, 30, 40, 50])

# Slicing returns a VIEW — modifying it changes the original!
view = arr[1:4]
view[0] = 999
print('Original after modifying view:', arr)   # [10, 999, 30, 40, 50]

# Boolean and fancy indexing return COPIES
arr = np.array([10, 20, 30, 40, 50])
copy = arr[[1, 2, 3]]
copy[0] = 999
print('Original after modifying copy:', arr)   # unchanged

In [ ]:
# When in doubt, force a copy with .copy()
arr = np.array([10, 20, 30, 40, 50])
safe = arr[1:4].copy()
safe[0] = 999
print('Original is safe:', arr)

## 6. Real example — filter funnel data

In [ ]:
df = pd.read_csv('data/raw/funnel_data.csv')
sessions = df['visita_landing'].to_numpy()
activations = df['activacion_tarjeta'].to_numpy()
channel = df['channel'].to_numpy()

# CVR end-to-end
cvr = activations / sessions * 100

# Top 10% performing rows by CVR
threshold = np.percentile(cvr, 90)
top_mask = cvr >= threshold

print(f'CVR p90 threshold: {threshold:.2f}%')
print(f'Rows above p90:    {top_mask.sum()}')
print('Channels in top:   ', np.unique(channel[top_mask]))

## Summary
| Pattern | Meaning |
|---|---|
| `arr[i]` | Element at position i |
| `arr[i, j]` | Element row i, col j (2D) |
| `arr[start:stop:step]` | Slice |
| `arr[:, j]` | Entire column j |
| `arr[mask]` | Boolean filter |
| `np.where(cond, a, b)` | Vectorized if-else |
| `arr[[0,2,4]]` | Fancy indexing |
| `.copy()` | Force a copy to avoid mutating original |

**Next:** `04_operations.ipynb` — arithmetic, broadcasting, and universal functions.